<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/04_analisis_resultados/L4S_08_sintesis_estado_del_arte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L4S_08 — Síntesis del Proyecto · Estado del Arte · Roadmap

**Proyecto:** Detección de Deslizamientos — Landslide4Sense  
**Propósito:** Este notebook no entrena modelos. Integra y analiza todos los resultados obtenidos,  
los contextualiza con la literatura reciente sobre el dataset, e identifica las brechas y oportunidades  
para llevar el proyecto a un nivel comparable con el estado del arte.

---
| Sección | Contenido |
|---------|-----------|
| 1 | Punto de partida — análisis inicial (2-Fold, datos parciales) |
| 2 | Estado actual — L4S 5-Fold (dataset completo, protocolo comparable) |
| 3 | Evolución del proyecto — qué mejoró y qué aprendimos |
| 4 | Revisión de literatura — papers sobre Landslide4Sense |
| 5 | Gap analysis — dónde estamos vs. estado del arte |
| 6 | Roadmap — qué incorporar para ser más competitivo |

> **No requiere GPU.** Todos los resultados se leen desde JSONs guardados en Drive.


In [ ]:
# ── Celda 0: Entorno ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'
ROOT = Path(DRIVE_PATH)
OUT_DIR = ROOT / 'results' / 'sintesis_estado_del_arte'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_json(rel, silent=False):
    p = ROOT / rel
    if p.exists():
        with open(p) as f: return json.load(f)
    if not silent: print(f'⚠️  No encontrado: {p}')
    return None

print(f'✅ Drive montado | Salida: {OUT_DIR}')


---
## 1. Punto de Partida — Análisis Inicial (2-Fold)

Resultados del primer ciclo de experimentación:  
- Dataset **parcial** (~1 500–2 000 muestras)  
- **2-Fold CV** con pocas épocas (3–10) — protocolo exploratorio, no comparable con literatura  
- Objetivo: verificar que el pipeline funcionaba y obtener una línea base rápida


In [ ]:
# ── Celda 1: Cargar resultados iniciales desde JSONs locales ─────────────────
# Estos archivos están en results/ (notebooks 03–06, protocolo 2-fold)

_cls_ini = load_json('results/classical_baselines/comparison_summary.json')
_rn_ini  = load_json('results/resnet50/kfold_summary.json')
_eff_ini = load_json('results/efficientnet_b4/kfold_summary.json')
_unet_ini= load_json('results/unet_resnet34/kfold_summary.json')
_rank_ini= load_json('results/final_summary.json')

# Construir tabla fase inicial
FASE1 = []
if _cls_ini:
    for m in _cls_ini.get('models', []):
        FASE1.append({
            'nombre': m['name'], 'tipo': 'Clásico',
            'f1': m['mean_f1'], 'std': m.get('std_f1', 0),
            'folds': _cls_ini.get('n_folds', 5),
            'n_samples': _cls_ini.get('n_samples', 1500),
            'epochs': '—',
        })
if _rn_ini:
    cfg = _rn_ini.get('config', {})
    f1s = [fo['best_f1'] for fo in _rn_ini.get('folds', [])]
    FASE1.append({
        'nombre': 'ResNet-50', 'tipo': 'Deep Learning',
        'f1': _rn_ini['mean_f1'], 'std': float(np.std(f1s)) if f1s else 0,
        'folds': cfg.get('n_folds', 2),
        'n_samples': 2000, 'epochs': cfg.get('epochs', '?'),
    })
if _eff_ini:
    FASE1.append({
        'nombre': 'EfficientNet-B4', 'tipo': 'Deep Learning',
        'f1': _eff_ini.get('mean_f1', 0), 'std': 0,
        'folds': 2, 'n_samples': 2000, 'epochs': '?',
    })
if _unet_ini:
    cfg = _unet_ini.get('config', {})
    f1s = [fo['best_f1'] for fo in _unet_ini.get('folds', [])]
    FASE1.append({
        'nombre': 'U-Net ResNet-34', 'tipo': 'Deep Learning',
        'f1': _unet_ini['mean_f1'], 'std': _unet_ini.get('std_f1', float(np.std(f1s)) if f1s else 0),
        'folds': cfg.get('n_folds', 2),
        'n_samples': 2000, 'epochs': cfg.get('epochs', '?'),
    })

FASE1.sort(key=lambda x: x['f1'], reverse=True)

print('='*72)
print(f'  FASE 1 — Análisis inicial')
print(f'  {"Modelo":<22} {"Tipo":<15} {"F1":>7} {"±std":>6}  {"Folds":>5}  {"Épocas":>6}')
print('='*72)
for m in FASE1:
    print(f'  {m["nombre"]:<22} {m["tipo"]:<15} {m["f1"]:>7.4f} {m["std"]:>6.4f}  '
          f'{str(m["folds"]):>5}  {str(m["epochs"]):>6}')
print('='*72)
print(f'  Nota: clásicos evaluados con ~{_cls_ini.get("n_samples",1500)} muestras, '
      f'DL con ~2 000 muestras, 2-Fold CV')


---
## 2. Estado Actual — L4S 5-Fold (Dataset Completo)

Resultados del protocolo comparable con la literatura:  
- **3 799 parches** (dataset completo)  
- **5-Fold CV** estratificado · 20 épocas · early stopping (patience=5)  
- Métricas: F1 patch-level (clásicos + clasificadores DL) y F1/Dice/IoU pixel-level (U-Net)


In [ ]:
# ── Celda 2: Cargar resultados L4S 5-Fold desde Drive ────────────────────────
_cls5  = load_json('results/comparable_literature/classicos_5fold/kfold5_summary.json')
_rn5   = load_json('results/comparable_literature/resnet50_5fold/kfold5_summary.json')
_eff5  = load_json('results/comparable_literatura/efficientnet_5fold/kfold5_summary.json')

_unet5_folds = []
for k in range(1, 6):
    d = load_json(f'results/comparable_literatura/unet_5fold/fold{k}_results.json', silent=True)
    if d: _unet5_folds.append(d)

# Construir tabla fase actual
FASE2 = []
if _cls5:
    for key, nombre in [('random_forest','Random Forest'),
                        ('svm_rbf','SVM (RBF)'),
                        ('logistic_regression','Log. Regression')]:
        d = _cls5['models'].get(key, {})
        if d:
            FASE2.append({
                'nombre': nombre, 'tipo': 'Clásico', 'nivel': 'patch',
                'f1': d['mean_f1'], 'std': d.get('std_f1', 0),
                'auc': d.get('mean_auc_roc', None),
            })
if _rn5:
    ag = _rn5['aggregate']
    FASE2.append({
        'nombre': 'ResNet-50', 'tipo': 'Deep Learning', 'nivel': 'patch',
        'f1': ag['mean_f1_thr05'], 'std': ag.get('std_f1_thr05', 0),
        'auc': ag.get('mean_auc_roc'),
    })
if _eff5:
    ag = _eff5['aggregate']
    FASE2.append({
        'nombre': 'EfficientNet-B4', 'tipo': 'Deep Learning', 'nivel': 'patch',
        'f1': ag['mean_f1_thr05'], 'std': ag.get('std_f1_thr05', 0),
        'auc': ag.get('mean_auc_roc'),
    })
if _unet5_folds:
    FASE2.append({
        'nombre': 'U-Net ResNet-34', 'tipo': 'Deep Learning', 'nivel': 'pixel',
        'f1':   float(np.mean([r['f1_pixel_thr05'] for r in _unet5_folds])),
        'std':  float(np.std( [r['f1_pixel_thr05'] for r in _unet5_folds])),
        'dice': float(np.mean([r['dice_thr05']     for r in _unet5_folds])),
        'iou':  float(np.mean([r['iou_thr05']      for r in _unet5_folds])),
        'auc':  float(np.mean([r['auc_roc']        for r in _unet5_folds])),
    })

FASE2.sort(key=lambda x: x['f1'], reverse=True)

print('='*72)
print(f'  FASE 2 — L4S 5-Fold · 3 799 parches · 20 épocas')
print(f'  {"Modelo":<22} {"Tipo":<15} {"F1":>7} {"±std":>6}  {"AUC-ROC":>8}  Nivel')
print('='*72)
for m in FASE2:
    auc_s = f'{m["auc"]:.4f}' if m.get('auc') else '   —   '
    print(f'  {m["nombre"]:<22} {m["tipo"]:<15} {m["f1"]:>7.4f} {m["std"]:>6.4f}  '
          f'{auc_s:>8}  {m["nivel"]}')
if _unet5_folds:
    u = [m for m in FASE2 if m["nombre"] == "U-Net ResNet-34"]
    if u:
        print(f'  U-Net adicional → Dice={u[0].get("dice","—"):.4f}  IoU={u[0].get("iou","—"):.4f}')
print('='*72)


---
## 3. Evolución del Proyecto

Comparación directa entre ambas fases para los modelos que participaron en los dos protocolos.


In [ ]:
# ── Celda 3: Gráfica de evolución Fase 1 → Fase 2 ───────────────────────────
# Modelos comparables entre fases
comunes = ['ResNet-50', 'EfficientNet-B4', 'U-Net ResNet-34',
           'Random Forest', 'SVM (RBF)', 'Log. Regression']

f1_fase1 = {m['nombre']: m['f1'] for m in FASE1}
f1_fase2 = {m['nombre']: m['f1'] for m in FASE2}

pares = [(n, f1_fase1.get(n), f1_fase2.get(n)) for n in comunes
         if f1_fase1.get(n) and f1_fase2.get(n)]
pares.sort(key=lambda x: x[2], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel izquierdo: barras agrupadas
ax = axes[0]
x = np.arange(len(pares))
w = 0.35
noms = [p[0] for p in pares]
v1   = [p[1] for p in pares]
v2   = [p[2] for p in pares]

b1 = ax.bar(x - w/2, v1, w, label='Fase 1 (2-Fold, parcial)', color='#93C5FD', edgecolor='white')
b2 = ax.bar(x + w/2, v2, w, label='Fase 2 (5-Fold, completo)', color='#1D4ED8', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(noms, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('F1-Score')
ax.set_title('Evolución F1-Score\nFase 1 → Fase 2', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', linestyle='--', alpha=0.3)
for bar, v in zip(list(b1)+list(b2), v1+v2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

# Panel derecho: delta (mejora/caída)
ax2 = axes[1]
deltas = [(p[0], (p[2]-p[1])*100) for p in pares]
noms_d = [d[0] for d in deltas]
vals_d = [d[1] for d in deltas]
cols_d = ['#16A34A' if v >= 0 else '#EF4444' for v in vals_d]
ax2.barh(range(len(noms_d)), vals_d, color=cols_d, edgecolor='white', height=0.6)
ax2.set_yticks(range(len(noms_d)))
ax2.set_yticklabels(noms_d, fontsize=9)
ax2.axvline(0, color='gray', lw=1)
ax2.set_xlabel('Δ F1 (puntos porcentuales)')
ax2.set_title('Cambio Fase 1 → Fase 2\n(verde = mejora)', fontsize=11, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', linestyle='--', alpha=0.3)
for i, v in enumerate(vals_d):
    ax2.text(v + (0.3 if v >= 0 else -0.3), i,
             f'{v:+.1f} pp', va='center', ha='left' if v >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'evolucion_fases.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAprendizajes clave de la evolución:')
mejoras = [(n, d) for n, d in deltas if d > 0]
caidas  = [(n, d) for n, d in deltas if d < 0]
for n, d in sorted(mejoras, key=lambda x: x[1], reverse=True):
    print(f'  ↑ {n:<22} mejoró {d:+.1f} pp con más datos y más épocas')
for n, d in sorted(caidas, key=lambda x: x[1]):
    print(f'  ↓ {n:<22} cayó  {d:+.1f} pp (requiere análisis adicional)')


---
## 4. Revisión de Literatura — Landslide4Sense

Recopilación de los principales trabajos publicados que usan este mismo dataset,  
con sus métodos, métricas reportadas y año de publicación.

> **Nota sobre comparabilidad:** Los trabajos evalúan a nivel de píxel (segmentación).  
> Nuestros clásicos y clasificadores DL operan a nivel de parche — no son directamente comparables  
> con los valores de segmentación de la literatura. Solo el U-Net es comparable píxel a píxel.


In [ ]:
# ── Celda 4: Tabla de literatura ─────────────────────────────────────────────
import matplotlib.patches as mpatches

LITERATURA = [
    # (Referencia, Año, Arquitectura, F1_pixel, IoU_pixel, Notas, URL)
    ('Ghorbanzadeh et al.',  2022, 'ResU-Net (baseline)',        0.7165, None,   'Paper original L4S · 11 modelos evaluados'),
    ('Ghorbanzadeh et al.',  2022, 'U-Net',                      0.6500, None,   'Baseline competición · punto de partida'),
    ('L4S Competition 1°',   2022, 'Swin Transformer (SwinLS)',  0.7390, None,   'Ganador competición 2022 · Swin + Mix-up'),
    ('L4S Competition 2°',   2022, 'SegFormer + hard mining',    0.7310, None,   'Segundo lugar · auto-entrenamiento'),
    ('L4S Competition 3°',   2022, 'U-Net + ensemble',           0.7280, None,   'Tercer lugar · ensamble de modelos'),
    ('Arxiv 2312.16717',     2023, 'U-Net + DNN híbrido',        0.7450, None,   'Fusión conv autoencoder + RNN'),
    ('Enhanced U-Net++ ',    2025, 'U-Net++ + Atención multhead', 0.8407, 0.7607, 'Mejor resultado publicado · residual + atención'),
    ('RMAU-Net',             2025, 'Residual Multihead Att U-Net',None,   0.6374, 'F1 detección=0.982 · IoU segmentación=0.637'),
    ('Multi-scale diff net', 2024, 'Multi-scale diff. features', 0.7600, 0.6500, 'IJGIS 2024 · características diferenciales'),
    ('SAR+optical fusion',   2024, 'Atención dual encoder',      0.7500, None,   '2024 · fusión explícita SAR vs óptico'),
]

# Este proyecto
NUESTRO_UNET = None
if _unet5_folds:
    NUESTRO_UNET = {
        'f1':   float(np.mean([r['f1_pixel_thr05'] for r in _unet5_folds])),
        'dice': float(np.mean([r['dice_thr05']      for r in _unet5_folds])),
        'iou':  float(np.mean([r['iou_thr05']       for r in _unet5_folds])),
    }

print('='*88)
print(f'  {"Referencia":<26} {"Año":>4}  {"Arquitectura":<30} {"F1 píxel":>9} {"IoU":>7}')
print('='*88)
for ref, year, arch, f1, iou, notes in LITERATURA:
    f1_s  = f'{f1:.4f}' if f1 else '  —   '
    iou_s = f'{iou:.4f}' if iou else '  —   '
    print(f'  {ref:<26} {year:>4}  {arch:<30} {f1_s:>9} {iou_s:>7}')
print('-'*88)
if NUESTRO_UNET:
    print(f'  {"★ Este proyecto (U-Net)":<26} {"2025":>4}  {"U-Net ResNet-34 · 5-Fold":<30} '
          f'{NUESTRO_UNET["f1"]:>9.4f} {NUESTRO_UNET["iou"]:>7.4f}  ← Nuestro resultado')
print('='*88)
print()
print('Fuentes:')
print('  · Ghorbanzadeh et al. 2022 — IEEE TGRS  doi:10.1109/TGRS.2022.3215209')
print('  · L4S Competition 2022 — arXiv:2209.02556')
print('  · Enhanced U-Net++ 2025 — ResearchGate 391077657')
print('  · RMAU-Net 2025 — arXiv:2507.11143')
print('  · Multi-scale diff. network 2024 — IJGIS doi:10.1080/17538947.2024.2441920')


In [ ]:
# ── Celda 5: Gráfica literatura — línea de tiempo de F1 ─────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

# Literatura con F1 disponible
lit_f1 = [(ref, year, arch, f1) for ref, year, arch, f1, iou, notes in LITERATURA if f1]
lit_f1.sort(key=lambda x: x[3], reverse=True)

y_lit  = [l[3] for l in lit_f1]
noms_l = [f'{l[0]} ({l[1]})' for l in lit_f1]
cols_l = ['#CBD5E1'] * len(lit_f1)

bars = ax.barh(range(len(noms_l)), y_lit, color=cols_l, edgecolor='white',
               height=0.6, label='Literatura')

# Nuestro resultado
if NUESTRO_UNET:
    ax.axvline(NUESTRO_UNET['f1'], color='#EF4444', lw=2.5, ls='--',
               label=f'Nuestro U-Net 5-Fold (F1={NUESTRO_UNET["f1"]:.4f})')
    # Zona de gap
    mejor_lit_f1 = max(y_lit)
    ax.axvspan(NUESTRO_UNET['f1'], mejor_lit_f1, alpha=0.08, color='#EF4444',
               label=f'Gap vs SOTA: {(mejor_lit_f1 - NUESTRO_UNET["f1"])*100:.1f} pp')

ax.set_yticks(range(len(noms_l)))
ax.set_yticklabels(noms_l, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('F1-Score (pixel-level)', fontsize=11)
ax.set_title('F1 pixel en Landslide4Sense — Literatura vs Este Proyecto\n'
             '(barras grises = publicaciones · línea roja = nuestro resultado)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(0.5, 1.0)
ax.grid(axis='x', linestyle='--', alpha=0.3)
for i, v in enumerate(y_lit):
    ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT_DIR / 'literatura_vs_proyecto.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: literatura_vs_proyecto.png')


---
## 5. Gap Analysis — Dónde Estamos vs. Estado del Arte

Identificación de las brechas técnicas entre nuestro proyecto y los mejores resultados publicados.


In [ ]:
# ── Celda 6: Radar chart — dimensiones de gap ───────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# Dimensiones evaluadas (0=mínimo, 1=máximo alcanzable)
DIMENSIONES = [
    'Protocolo\nevaluación',
    'Arquitectura\nmodelo',
    'Augmentación\ndatos',
    'Análisis\nerror',
    'Evaluación\ntest set',
    'Ensamble\nmodelos',
    'Fusión\nSAR/Óptico',
    'Explicabilidad',
]

# Puntuación 0-1 de nuestro proyecto vs SOTA
NUESTRO  = [0.9, 0.5, 0.5, 0.7, 0.1, 0.2, 0.3, 0.6]
SOTA_REF = [1.0, 1.0, 0.9, 0.8, 1.0, 0.8, 0.9, 0.7]

NOTAS = [
    '5-Fold CV completo ✅ · sin evaluación en test set oficial ⚠️',
    'U-Net ResNet-34 básico · sin atención, sin transformer',
    'Solo flips + rot90 · sin mix-up, sin elastic, sin hard mining',
    'Error maps, FP/FN, incertidumbre ✅ · sin análisis geoespacial',
    'No evaluado en TestData oficial L4S ❌',
    'No implementado · literatura reporta +2-5 pp con ensamble',
    'Concatenación directa 14 ch · sin fusión explícita por modalidad',
    'Grad-CAM ✅ · sin SHAP para DL, sin saliency maps',
]

N = len(DIMENSIONES)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

nos_plot  = NUESTRO + NUESTRO[:1]
sota_plot = SOTA_REF + SOTA_REF[:1]

fig, (ax_radar, ax_bar) = plt.subplots(1, 2, figsize=(15, 6),
                                        subplot_kw={'projection': None})

# Radar
ax_r = fig.add_subplot(121, projection='polar')
ax_r.plot(angles, sota_plot, 'o-', lw=2, color='#CBD5E1', label='Referencia SOTA')
ax_r.fill(angles, sota_plot, alpha=0.1, color='#CBD5E1')
ax_r.plot(angles, nos_plot,  'o-', lw=2.5, color='#1D4ED8', label='Este proyecto')
ax_r.fill(angles, nos_plot,  alpha=0.2, color='#1D4ED8')
ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(DIMENSIONES, fontsize=8.5)
ax_r.set_ylim(0, 1)
ax_r.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_r.set_yticklabels(['0.25','0.5','0.75','1.0'], fontsize=7)
ax_r.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax_r.set_title('Madurez del proyecto\nvs referencia SOTA', fontsize=11,
               fontweight='bold', pad=20)

# Barras de gap
ax2 = fig.add_subplot(122)
gaps = [s - n for s, n in zip(SOTA_REF, NUESTRO)]
dim_short = [d.replace('\n', ' ') for d in DIMENSIONES]
cols_g = ['#EF4444' if g > 0.4 else '#F59E0B' if g > 0.2 else '#16A34A'
          for g in gaps]
ax2.barh(range(N), gaps, color=cols_g, edgecolor='white', height=0.6)
ax2.set_yticks(range(N))
ax2.set_yticklabels(dim_short, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('Gap vs SOTA (0=paridad · 1=máximo)')
ax2.set_title('Brechas por dimensión\n(rojo=crítico · naranja=moderado · verde=cubierto)',
              fontsize=11, fontweight='bold')
ax2.axvline(0.4, color='#EF4444', lw=1, ls='--', alpha=0.5)
ax2.axvline(0.2, color='#F59E0B', lw=1, ls='--', alpha=0.5)
ax2.grid(axis='x', linestyle='--', alpha=0.3)
for i, (v, nota) in enumerate(zip(gaps, NOTAS)):
    ax2.text(v + 0.01, i, f'{v:.1f}', va='center', fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT_DIR / 'gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDetalle de gaps:')
for dim, gap, nota in sorted(zip(dim_short, gaps, NOTAS), key=lambda x: x[1], reverse=True):
    nivel = '🔴 CRÍTICO' if gap > 0.4 else '🟡 MODERADO' if gap > 0.2 else '🟢 CUBIERTO'
    print(f'  {nivel}  {dim:<28} gap={gap:.1f}')
    print(f'           {nota}')


---
## 6. Roadmap — Qué Incorporar para Ser Más Competitivo

Acciones priorizadas por impacto esperado en F1 píxel y viabilidad de implementación.  
Cada ítem incluye la referencia de la literatura que lo valida.


In [ ]:
# ── Celda 7: Roadmap visual ───────────────────────────────────────────────────
ROADMAP = [
    # (Prioridad, Acción, Δ_F1_esperado, Esfuerzo, Referencia)
    (1, 'Evaluar en TestData oficial L4S',
        '+comparabilidad directa', 'Bajo',
        'Protocolo competición 2022'),
    (2, 'Añadir mecanismo de atención al decoder U-Net\n(CBAM o self-attention)',
        '+3–8 pp F1 pixel', 'Medio',
        'Enhanced U-Net++ 2025: F1=0.84'),
    (3, 'Mix-up augmentation + hard example mining',
        '+2–4 pp F1 pixel', 'Medio',
        'Ganador L4S Competition 2022'),
    (4, 'Ensamble de folds (avg probabilidades)',
        '+1–3 pp F1 pixel', 'Bajo',
        'Competition 3er lugar · literatura ensamble'),
    (5, 'Test-Time Augmentation (TTA)\n(flips + rotaciones en inferencia)',
        '+1–2 pp F1 pixel', 'Bajo',
        'Práctica estándar en SOTA segmentación'),
    (6, 'Aumentar pos_weight en loss BCE (>1.0)\npara compensar desbalance clase',
        '+1–3 pp F1 pixel', 'Bajo',
        'Análisis interno: pos_weight=0.703 suboptimal'),
    (7, 'Probar encoder Swin Transformer o SegFormer',
        '+4–8 pp F1 pixel', 'Alto',
        'SwinLS L4S: F1=0.739 · RMAU-Net 2025'),
    (8, 'Fusión explícita por modalidad\n(rama SAR separada + rama óptica)',
        '+2–5 pp F1 pixel', 'Alto',
        'SAR+optical fusion 2024, ablación NB01'),
    (9, 'Análisis de transferibilidad a Colombia\n(fine-tuning con datos sintéticos)',
        'Generalización', 'Alto',
        'Objetivo final del proyecto'),
]

fig, ax = plt.subplots(figsize=(14, 7))

EFFORT_COL = {'Bajo': '#16A34A', 'Medio': '#F59E0B', 'Alto': '#EF4444'}
y_labels = []

for i, (prio, accion, delta, esfuerzo, ref) in enumerate(ROADMAP):
    color = EFFORT_COL[esfuerzo]
    y = len(ROADMAP) - i
    ax.barh(y, 1, left=prio-1, color=color, alpha=0.75, edgecolor='white', height=0.6)
    label = accion.replace('\n', ' ')
    ax.text(prio - 0.95, y, f'{prio}. {label}', va='center', fontsize=8.5,
            fontweight='bold')
    ax.text(prio - 0.95, y - 0.28, f'  {delta} | Ref: {ref}',
            va='center', fontsize=7.5, color='#374151')
    y_labels.append(f'P{prio}')

ax.set_xlim(0, len(ROADMAP))
ax.set_ylim(0, len(ROADMAP) + 1)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title('Roadmap de mejoras — ordenado por prioridad\n'
             '(verde=esfuerzo bajo · naranja=medio · rojo=alto)',
             fontsize=12, fontweight='bold')

patches = [mpatches.Patch(color=c, label=f'Esfuerzo {e}')
           for e, c in EFFORT_COL.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'roadmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Roadmap detallado:')
print('='*75)
for prio, accion, delta, esfuerzo, ref in ROADMAP:
    print(f'  P{prio} [{esfuerzo:<5}] {accion.replace(chr(10)," "):<50} → {delta}')
    print(f'         Referencia: {ref}')
print('='*75)


---
## 7. Resumen Ejecutivo


In [ ]:
# ── Celda 8: Resumen ejecutivo ────────────────────────────────────────────────
mejor_f1_ini  = max(m['f1'] for m in FASE1) if FASE1 else 0
mejor_f1_act  = max(m['f1'] for m in FASE2) if FASE2 else 0
mejor_mod_ini = max(FASE1, key=lambda x: x['f1'])['nombre'] if FASE1 else '—'
mejor_mod_act = max(FASE2, key=lambda x: x['f1'])['nombre'] if FASE2 else '—'
sota_f1_pixel = 0.8407  # Enhanced U-Net++ 2025

print('='*65)
print('  RESUMEN EJECUTIVO — DETECCIÓN DE DESLIZAMIENTOS L4S')
print('='*65)
print(f'\n  FASE 1 (inicio):')
print(f'    Mejor modelo : {mejor_mod_ini}  F1={mejor_f1_ini:.4f}')
print(f'    Protocolo    : 2-Fold · ~2 000 muestras · pocas épocas')
print(f'    Objetivo     : verificar pipeline y obtener baseline rápido')
print(f'\n  FASE 2 (estado actual):')
print(f'    Mejor modelo : {mejor_mod_act}  F1={mejor_f1_act:.4f}')
print(f'    Protocolo    : 5-Fold · 3 799 muestras · 20 épocas + early stop')
print(f'    Objetivo     : resultados comparables con literatura')
if NUESTRO_UNET:
    print(f'\n  U-Net (segmentación pixel-level):')
    print(f'    F1 pixel     : {NUESTRO_UNET["f1"]:.4f}')
    print(f'    Dice         : {NUESTRO_UNET["dice"]:.4f}')
    print(f'    IoU          : {NUESTRO_UNET["iou"]:.4f}')
    gap = (sota_f1_pixel - NUESTRO_UNET["f1"]) * 100
    print(f'    Gap vs SOTA  : {gap:.1f} pp  (SOTA: Enhanced U-Net++ F1=0.8407)')
print(f'\n  Próximos pasos de mayor impacto:')
print(f'    1. Evaluar en TestData oficial → comparabilidad directa')
print(f'    2. Añadir atención al decoder U-Net → ~+3-8 pp F1 pixel')
print(f'    3. Ensamble de folds + TTA → ~+2-4 pp con esfuerzo bajo')
print(f'    4. Corregir pos_weight > 1.0 → mejora calibración del modelo')
print(f'\n  Figuras guardadas en: {OUT_DIR}')
print(f'    evolucion_fases.png · literatura_vs_proyecto.png')
print(f'    gap_analysis.png    · roadmap.png')
print('='*65)
